# FNO2d - the Fourier Neural Operator

*Walkthrough notebook for `neural_operators/models/fno.py`.*

The full model, following the paper's Eq. (6):

$$\mathcal G_\theta=Q\circ\sigma(W_{T-1}+\mathcal K_{T-1}+b_{T-1})\circ\dots\circ\sigma(W_0+\mathcal K_0+b_0)\circ P$$

| Symbol | What it is | In code |
|---|---|---|
| $P$ | lifts each point's input $(a(x),x,y)$ to a wide hidden vector | `self.lift` |
| $\mathcal K_t$ | the non-local integral operator | `SpectralConv2d` |
| $W_t$ | pointwise (local) linear map, added next to $\mathcal K_t$ | 1x1 `Conv2d` |
| $Q$ | projects the hidden vector back to the output function | `project1`, `project2` |

Everything except $\mathcal K_t$ acts independently per grid point, and $\mathcal K_t$ only keeps a fixed number
of Fourier modes, so the *same weights* work on any grid resolution.

## Setup

The first cell makes the repository importable when this notebook is opened from `notebooks/source/`; the second holds the module's imports.

In [ ]:
# Make the repo root importable so `import neural_operators` works from notebooks/source/.
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / "neural_operators").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from neural_operators.layers.spectral_conv import SpectralConv2d

## One Fourier layer

`spectral_conv(x) + local_linear(x)` is $\mathcal K_t v+W_t v$: the global (Fourier) path plus the local
(pointwise) path. GELU is applied after every block except the last.

In [ ]:
class FNOBlock2d(nn.Module):
    def __init__(self, width, modes1, modes2, activation=True):
        super().__init__()
        self.spectral_conv = SpectralConv2d(width, width, modes1, modes2)
        self.local_linear = nn.Conv2d(width, width, kernel_size=1)  # W_t (pointwise)
        self.activation = activation

    def forward(self, x):
        x = self.spectral_conv(x) + self.local_linear(x)
        if self.activation:
            x = F.gelu(x)
        return x

## The whole network

Three details worth understanding in `forward`:

1. **Coordinate trick** - `get_grid` appends the $(x,y)$ position of every pixel to the input, giving the
   network an explicit notion of location.
2. **Channels-last vs channels-first** - `nn.Linear` wants channels last, `Conv2d`/FFT layers want channels
   first, hence the two `permute` calls.
3. **Output shape** - the final `squeeze(-1)` returns one scalar field per sample.

In [ ]:
class FNO2d(nn.Module):
    def __init__(
        self,
        in_channels=3,   # e.g. (a(x), x_coord, y_coord) after the coordinate-preprocessing trick
        out_channels=1,
        width=32,
        modes1=12,
        modes2=12,
        n_layers=4,
    ):
        super().__init__()
        self.width = width

        self.lift = nn.Linear(in_channels, width)  # P: pointwise lift, applied per-pixel
        self.blocks = nn.ModuleList(
            [
                FNOBlock2d(width, modes1, modes2, activation=(t < n_layers - 1))
                for t in range(n_layers)
            ]
        )
        self.project1 = nn.Linear(width, 128)  # Q: pointwise projection (two-layer MLP)
        self.project2 = nn.Linear(128, out_channels)

    @staticmethod
    def get_grid(shape, device):
        batch, size_x, size_y = shape[0], shape[1], shape[2]
        gx = torch.linspace(0, 1, size_x, device=device).reshape(1, size_x, 1, 1).repeat(batch, 1, size_y, 1)
        gy = torch.linspace(0, 1, size_y, device=device).reshape(1, 1, size_y, 1).repeat(batch, size_x, 1, 1)
        return torch.cat((gx, gy), dim=-1)  # (batch, size_x, size_y, 2)

    def forward(self, a):
        # a: (batch, size_x, size_y) or (batch, size_x, size_y, in_channels-2) raw input function.
        if a.dim() == 3:
            a = a.unsqueeze(-1)
        grid = self.get_grid(a.shape, a.device)
        x = torch.cat((a, grid), dim=-1)  # append (x, y) coordinates -- see "Preprocessing" in the paper

        x = self.lift(x)               # (batch, size_x, size_y, width)
        x = x.permute(0, 3, 1, 2)      # -> (batch, width, size_x, size_y) for SpectralConv2d/Conv2d

        for block in self.blocks:
            x = block(x)

        x = x.permute(0, 2, 3, 1)      # -> (batch, size_x, size_y, width)
        x = F.gelu(self.project1(x))
        x = self.project2(x)
        return x.squeeze(-1)           # (batch, size_x, size_y)

## Try it

Build the default model, count its parameters (the saved result in `results/fno_summary.json` reports 1,188,353), and run it at several resolutions.

In [ ]:
from neural_operators.utils import count_params

model = FNO2d()          # defaults: width 32, 12x12 Fourier modes, 4 layers
print("parameters:", count_params(model))

a = torch.randn(2, 64, 64)               # a batch of 2 input fields on a 64x64 grid
print("64x64 in ->", tuple(model(a).shape))

In [ ]:
# Discretization invariance: no retraining, any grid.
for n in (32, 64, 96, 128):
    print(f"{n:>3}x{n:<3} in -> {tuple(model(torch.randn(1, n, n)).shape)}")